In [1]:
# Load the raw Bisnode panel (data is in the same folder as this notebook)

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

# Load data (same folder as this .ipynb)
data = pd.read_csv("cs_bisnode_panel.csv")

# Keep the required panel years
data = data.query("2010 <= year <= 2015").copy()

print("Shape:", data.shape)
print("Years in data:", sorted(data["year"].unique()))
data.head()


Shape: (167606, 48)
Years in data: [2010, 2011, 2012, 2013, 2014, 2015]


,comp_id,begin,end,COGS,amort,curr_assets,curr_liab,extra_exp,extra_inc,extra_profit_loss,...,gender,origin,nace_main,ind2,ind,urban_m,region_m,founded_date,exit_date,labor_avg
5,1001034.0,2010-01-01,2010-12-31,NaN,177.777771,2096.296387,19629.628906,0.0,0.0,0.0,...,mix,Domestic,5630.0,56.0,3.0,1,Central,1990-11-19,NaN,0.083333
6,1001034.0,2011-01-01,2011-12-31,NaN,155.555557,3607.407471,22555.554688,0.0,0.0,0.0,...,mix,Domestic,5630.0,56.0,3.0,1,Central,1990-11-19,NaN,0.083333
7,1001034.0,2012-01-01,2012-12-31,NaN,140.740738,148.148148,21429.628906,0.0,0.0,0.0,...,mix,Domestic,5630.0,56.0,3.0,1,Central,1990-11-19,NaN,0.083333
8,1001034.0,2013-01-01,2013-12-31,NaN,140.740738,140.740738,21851.851562,0.0,0.0,0.0,...,mix,Domestic,5630.0,56.0,3.0,1,Central,1990-11-19,NaN,NaN
9,1001034.0,2014-01-01,2014-12-31,NaN,29.629629,144.444443,22340.740234,0.0,0.0,0.0,...,mix,Domestic,5630.0,56.0,3.0,1,Central,1990-11-19,NaN,NaN


In [2]:
# Basic cleaning + sales variable needed for growth

# Make sales usable (no negatives or zeros)
data["sales_pos"] = np.where(
    data["sales"].isna(),
    np.nan,
    np.where(data["sales"] <= 0, 1, data["sales"])
)

# Log sales
data["ln_sales"] = np.log(data["sales_pos"])

data[["year", "comp_id", "sales", "ln_sales"]].head()


,year,comp_id,sales,ln_sales
5,2010,1001034.0,9929.629883,9.203278
6,2011,1001034.0,0.000000,0.000000
7,2012,1001034.0,0.000000,0.000000
8,2013,1001034.0,0.000000,0.000000
9,2014,1001034.0,0.000000,0.000000


In [3]:
# Create future sales and growth measures (1-year and 2-year)

# Future log sales
data["ln_sales_lead1"] = data.groupby("comp_id")["ln_sales"].shift(-1)  # t+1
data["ln_sales_lead2"] = data.groupby("comp_id")["ln_sales"].shift(-2)  # t+2

# Log-growth (≈ percentage growth)
data["g1_ln_sales"] = data["ln_sales_lead1"] - data["ln_sales"]  # 2013 vs 2012
data["g2_ln_sales"] = data["ln_sales_lead2"] - data["ln_sales"]  # 2014 vs 2012

# Quick sanity check
data[["year", "comp_id", "g1_ln_sales", "g2_ln_sales"]].head(10)


,year,comp_id,g1_ln_sales,g2_ln_sales
5,2010,1001034.0,-9.203278,-9.203278
6,2011,1001034.0,0.000000,0.000000
7,2012,1001034.0,0.000000,0.000000
8,2013,1001034.0,0.000000,0.000000
9,2014,1001034.0,0.000000,NaN
10,2015,1001034.0,NaN,NaN
12,2010,1001541.0,0.000000,7.093159
13,2011,1001541.0,7.093159,8.622554
14,2012,1001541.0,1.529395,1.401562
15,2013,1001541.0,-0.127833,-0.127833


In [4]:
# CELL 4 — Lock the base-year sample (2012) and define FAST GROWTH target

# Base year sample: alive firms in 2012 with reasonable size
df = data.query("year == 2012").copy()

# Basic size filter (same spirit as class)
df = df.query("sales_pos >= 1_000 & sales_pos <= 10_000_000")

# Choose horizon (2-year recommended)
growth_var = "g2_ln_sales"   # change to "g1_ln_sales" for 1-year growth

# Keep firms with observed growth
df = df.dropna(subset=[growth_var])

# Define fast growth as top 10% growers
threshold = df[growth_var].quantile(0.90)
df["fast_growth"] = (df[growth_var] >= threshold).astype(int)

print("Fast-growth share:", df["fast_growth"].mean())
print("Growth threshold:", threshold)

df[["comp_id", "sales_pos", growth_var, "fast_growth"]].head()


Fast-growth share: 0.10000528569163275
Growth threshold: 0.9775200784470454


,comp_id,sales_pos,g2_ln_sales,fast_growth
14,1001541.0,1.203704e+03,1.401562,1
23,1002029.0,1.136515e+06,-1.656969,0
56,1011889.0,4.336667e+05,0.086522,0
68,1014183.0,1.297296e+05,-0.249819,0
80,1018301.0,6.722222e+03,0.300819,0


In [5]:
# CELL 5 — Keep features + save the modeling dataset

# Minimal, safe feature set (you can add later)
keep_vars = [
    "comp_id",
    "year",
    "fast_growth",
    "sales_pos",
    "ln_sales",
    "g1_ln_sales",
    "g2_ln_sales",
    "age",
    "foreign",
    "labor_avg",
    "profit_loss_year",
    "material_exp",
    "fixed_assets",
    "curr_assets",
    "liq_assets",
    "share_eq",
    "ind",          # industry (needed later for manufacturing vs services)
    "region_m",
]

# Keep only existing columns (robust to missing vars)
keep_vars = [c for c in keep_vars if c in df.columns]
df_model = df[keep_vars].copy()

print("Final modeling shape:", df_model.shape)
print(df_model["fast_growth"].value_counts(normalize=True))

# Save — this is YOUR produced dataset
df_model.to_csv("bisnode_fastgrowth_2012.csv", index=False)

df_model.head()


Final modeling shape: (18919, 17)
fast_growth
0    0.899995
1    0.100005
Name: proportion, dtype: float64


,comp_id,year,fast_growth,sales_pos,ln_sales,g1_ln_sales,g2_ln_sales,foreign,labor_avg,profit_loss_year,material_exp,fixed_assets,curr_assets,liq_assets,share_eq,ind,region_m
14,1001541.0,2012,1,1.203704e+03,7.093159,1.529395,1.401562,0.0,NaN,-7722.222168,8351.851562,190566.671875,9629.629883,9048.148438,1.912630e+05,3.0,Central
23,1002029.0,2012,0,1.136515e+06,13.943477,-1.155013,-1.656969,0.0,0.458333,9722.222656,984270.375000,23459.259766,203885.187500,15077.777344,9.314445e+04,2.0,East
56,1011889.0,2012,0,4.336667e+05,12.980031,0.019109,0.086522,0.0,1.621212,96751.851562,204659.265625,933574.062500,160166.671875,131766.671875,1.071011e+06,3.0,West
68,1014183.0,2012,0,1.297296e+05,11.773208,-0.110044,-0.249819,0.0,0.715278,-2351.851807,66744.445312,118229.632812,199903.703125,18585.185547,3.098852e+05,3.0,Central
80,1018301.0,2012,0,6.722222e+03,8.813174,0.228529,0.300819,0.0,0.152778,-1107.407349,3325.926025,52155.554688,1811.111084,1314.814819,1.078148e+04,3.0,Central


In [8]:
# CELL 6 (fix) — make X numeric-only, then LOGIT 5-fold CV

from sklearn.model_selection import KFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, brier_score_loss
import numpy as np
import pandas as pd

df = pd.read_csv("bisnode_fastgrowth_2012.csv")

y = df["fast_growth"]

# Drop ids, then keep ONLY numeric columns (fixes your error)
X = df.drop(columns=["fast_growth", "comp_id", "year"], errors="ignore")
X = X.select_dtypes(include=[np.number])

# Mean-impute numeric columns
X = X.fillna(X.mean(numeric_only=True))

kf = KFold(n_splits=5, shuffle=True, random_state=42)
aucs, briers = [], []

for tr, te in kf.split(X):
    Xtr, Xte = X.iloc[tr], X.iloc[te]
    ytr, yte = y.iloc[tr], y.iloc[te]

    logit = LogisticRegression(max_iter=2000)
    logit.fit(Xtr, ytr)

    p = logit.predict_proba(Xte)[:, 1]
    aucs.append(roc_auc_score(yte, p))
    briers.append(brier_score_loss(yte, p))

print("LOGIT — CV AUC (mean):", round(np.mean(aucs), 3))
print("LOGIT — CV Brier (mean):", round(np.mean(briers), 3))
print("Features used:", X.shape[1])


LOGIT — CV AUC (mean): 0.999
LOGIT — CV Brier (mean): 0.008
Features used: 13


In [9]:
# CELL 7 — PART I: Probability prediction with RANDOM FOREST (5-fold CV)

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import KFold
from sklearn.metrics import roc_auc_score, brier_score_loss
import numpy as np
import pandas as pd

df = pd.read_csv("bisnode_fastgrowth_2012.csv")

y = df["fast_growth"]

X = df.drop(columns=["fast_growth", "comp_id", "year"], errors="ignore")
X = X.select_dtypes(include=[np.number])
X = X.fillna(X.mean(numeric_only=True))

kf = KFold(n_splits=5, shuffle=True, random_state=42)

aucs, briers = [], []

for tr, te in kf.split(X):
    Xtr, Xte = X.iloc[tr], X.iloc[te]
    ytr, yte = y.iloc[tr], y.iloc[te]

    rf = RandomForestClassifier(
        n_estimators=500,
        random_state=42,
        max_features="sqrt",
        min_samples_leaf=20,
    )
    rf.fit(Xtr, ytr)

    p = rf.predict_proba(Xte)[:, 1]

    aucs.append(roc_auc_score(yte, p))
    briers.append(brier_score_loss(yte, p))

print("RF — CV AUC (mean):", round(np.mean(aucs), 3))
print("RF — CV Brier (mean):", round(np.mean(briers), 3))
print("Features used:", X.shape[1])


RF — CV AUC (mean): 1.0
RF — CV Brier (mean): 0.001
Features used: 13


In [10]:
# CELL 8 — PART I: Probability prediction with GRADIENT BOOSTING (3rd model, 5-fold CV)

from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import KFold
from sklearn.metrics import roc_auc_score, brier_score_loss
import numpy as np
import pandas as pd

df = pd.read_csv("bisnode_fastgrowth_2012.csv")

y = df["fast_growth"]

X = df.drop(columns=["fast_growth", "comp_id", "year"], errors="ignore")
X = X.select_dtypes(include=[np.number])
X = X.fillna(X.mean(numeric_only=True))

kf = KFold(n_splits=5, shuffle=True, random_state=42)

aucs, briers = [], []

for tr, te in kf.split(X):
    Xtr, Xte = X.iloc[tr], X.iloc[te]
    ytr, yte = y.iloc[tr], y.iloc[te]

    gb = GradientBoostingClassifier(random_state=42)
    gb.fit(Xtr, ytr)

    p = gb.predict_proba(Xte)[:, 1]

    aucs.append(roc_auc_score(yte, p))
    briers.append(brier_score_loss(yte, p))

print("GB — CV AUC (mean):", round(np.mean(aucs), 3))
print("GB — CV Brier (mean):", round(np.mean(briers), 3))
print("Features used:", X.shape[1])


GB — CV AUC (mean): 1.0
GB — CV Brier (mean): 0.0
Features used: 13


In [11]:
# CELL 9 — PART II: Classification with LOSS FUNCTION (pick best model)

from sklearn.model_selection import KFold
from sklearn.metrics import confusion_matrix
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

# Load data
df = pd.read_csv("bisnode_fastgrowth_2012.csv")
y = df["fast_growth"]

X = df.drop(columns=["fast_growth", "comp_id", "year"], errors="ignore")
X = X.select_dtypes(include=[np.number])
X = X.fillna(X.mean(numeric_only=True))

# Loss function (business decision)
FP_COST = 1     # false positive cost
FN_COST = 10    # false negative cost (missed fast-grower is expensive)

def expected_loss(y_true, y_prob, thr, FP=1, FN=10):
    y_hat = (y_prob >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_hat, labels=[0,1]).ravel()
    return (fp * FP + fn * FN) / len(y_true)

def best_threshold(y_true, y_prob, FP=1, FN=10):
    grid = np.linspace(0.01, 0.99, 99)
    losses = [expected_loss(y_true, y_prob, t, FP, FN) for t in grid]
    i = int(np.argmin(losses))
    return grid[i], losses[i]

models = {
    "Logit": LogisticRegression(max_iter=2000),
    "RF": RandomForestClassifier(
        n_estimators=500, random_state=42,
        max_features="sqrt", min_samples_leaf=20
    ),
    "GB": GradientBoostingClassifier(random_state=42),
}

kf = KFold(n_splits=5, shuffle=True, random_state=42)

results = []

for name, model in models.items():
    fold_losses = []
    fold_thresholds = []

    for tr, te in kf.split(X):
        Xtr, Xte = X.iloc[tr], X.iloc[te]
        ytr, yte = y.iloc[tr], y.iloc[te]

        model.fit(Xtr, ytr)
        p = model.predict_proba(Xte)[:, 1]

        thr, loss = best_threshold(yte.values, p, FP_COST, FN_COST)
        fold_losses.append(loss)
        fold_thresholds.append(thr)

    results.append({
        "model": name,
        "avg_expected_loss": np.mean(fold_losses),
        "avg_optimal_threshold": np.mean(fold_thresholds)
    })

pd.DataFrame(results).sort_values("avg_expected_loss")


,model,avg_expected_loss,avg_optimal_threshold
1,RF,0.000159,0.27
2,GB,0.000634,0.01
0,Logit,0.020826,0.17


In [12]:
# CELL 10 — PART III: Confusion matrix on a holdout set (for discussion)

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, roc_auc_score, brier_score_loss
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

# Reload data
df = pd.read_csv("bisnode_fastgrowth_2012.csv")
y = df["fast_growth"]

X = df.drop(columns=["fast_growth", "comp_id", "year"], errors="ignore")
X = X.select_dtypes(include=[np.number])
X = X.fillna(X.mean(numeric_only=True))

# Train / holdout split
X_tr, X_ho, y_tr, y_ho = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 👉 Use the WINNING model from Cell 9
best_model = RandomForestClassifier(
    n_estimators=500,
    random_state=42,
    max_features="sqrt",
    min_samples_leaf=20,
)

best_model.fit(X_tr, y_tr)
p_ho = best_model.predict_proba(X_ho)[:, 1]

# Use average optimal threshold from CV (adjust if your winner was not RF)
OPT_THR = 0.5   # replace with avg_optimal_threshold from Cell 9 for your winning model

y_hat = (p_ho >= OPT_THR).astype(int)

# Confusion matrix
cm = confusion_matrix(y_ho, y_hat, labels=[0,1])
cm_df = pd.DataFrame(
    cm,
    index=["Actual: not fast-growth", "Actual: fast-growth"],
    columns=["Pred: not fast-growth", "Pred: fast-growth"],
)

print(cm_df)
print("\nAUC (holdout):", round(roc_auc_score(y_ho, p_ho), 3))
print("Brier (holdout):", round(brier_score_loss(y_ho, p_ho), 3))


                         Pred: not fast-growth  Pred: fast-growth
Actual: not fast-growth                   3406                  0
Actual: fast-growth                          0                378

AUC (holdout): 1.0
Brier (holdout): 0.001


In [13]:
# CELL 11 — Remove future-information leakage

df = pd.read_csv("bisnode_fastgrowth_2012.csv")

# Target
y = df["fast_growth"]

# Drop identifiers AND growth variables (these caused leakage)
X = df.drop(
    columns=[
        "fast_growth",
        "comp_id",
        "year",
        "g1_ln_sales",
        "g2_ln_sales",
    ],
    errors="ignore"
)

# Keep numeric only
X = X.select_dtypes(include=[np.number])

# Impute
X = X.fillna(X.mean(numeric_only=True))

print("Features used:", X.columns.tolist())
print("N features:", X.shape[1])


Features used: ['sales_pos', 'ln_sales', 'foreign', 'labor_avg', 'profit_loss_year', 'material_exp', 'fixed_assets', 'curr_assets', 'liq_assets', 'share_eq', 'ind']
N features: 11


In [14]:
# CELL 12 — PART I (CORRECTED): Probability prediction after removing leakage

from sklearn.model_selection import KFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, brier_score_loss
import numpy as np
import pandas as pd

# Load cleaned data (no future info)
df = pd.read_csv("bisnode_fastgrowth_2012.csv")

y = df["fast_growth"]

X = df.drop(
    columns=[
        "fast_growth",
        "comp_id",
        "year",
        "g1_ln_sales",
        "g2_ln_sales",
    ],
    errors="ignore"
)
X = X.select_dtypes(include=[np.number])
X = X.fillna(X.mean(numeric_only=True))

kf = KFold(n_splits=5, shuffle=True, random_state=42)

results = []

# ----- LOGIT -----
aucs, briers = [], []
for tr, te in kf.split(X):
    logit = LogisticRegression(max_iter=2000)
    logit.fit(X.iloc[tr], y.iloc[tr])
    p = logit.predict_proba(X.iloc[te])[:, 1]
    aucs.append(roc_auc_score(y.iloc[te], p))
    briers.append(brier_score_loss(y.iloc[te], p))

results.append({
    "model": "Logit",
    "cv_auc": np.mean(aucs),
    "cv_brier": np.mean(briers),
})

# ----- RANDOM FOREST -----
aucs, briers = [], []
for tr, te in kf.split(X):
    rf = RandomForestClassifier(
        n_estimators=500,
        random_state=42,
        max_features="sqrt",
        min_samples_leaf=20,
    )
    rf.fit(X.iloc[tr], y.iloc[tr])
    p = rf.predict_proba(X.iloc[te])[:, 1]
    aucs.append(roc_auc_score(y.iloc[te], p))
    briers.append(brier_score_loss(y.iloc[te], p))

results.append({
    "model": "Random Forest",
    "cv_auc": np.mean(aucs),
    "cv_brier": np.mean(briers),
})

pd.DataFrame(results)


,model,cv_auc,cv_brier
0,Logit,0.728184,0.083570
1,Random Forest,0.778282,0.078431


In [15]:
# CELL 13 — PART II (CORRECTED): Loss function + optimal threshold (NO leakage)

from sklearn.model_selection import KFold
from sklearn.metrics import confusion_matrix
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Load data
df = pd.read_csv("bisnode_fastgrowth_2012.csv")
y = df["fast_growth"]

X = df.drop(
    columns=[
        "fast_growth",
        "comp_id",
        "year",
        "g1_ln_sales",
        "g2_ln_sales",
    ],
    errors="ignore"
)
X = X.select_dtypes(include=[np.number])
X = X.fillna(X.mean(numeric_only=True))

# Business loss assumptions
FP_COST = 1
FN_COST = 10

def expected_loss(y_true, y_prob, thr, FP=1, FN=10):
    y_hat = (y_prob >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_hat, labels=[0,1]).ravel()
    return (fp * FP + fn * FN) / len(y_true)

def best_threshold(y_true, y_prob, FP=1, FN=10):
    grid = np.linspace(0.01, 0.99, 99)
    losses = [expected_loss(y_true, y_prob, t, FP, FN) for t in grid]
    i = int(np.argmin(losses))
    return grid[i], losses[i]

models = {
    "Logit": LogisticRegression(max_iter=2000),
    "Random Forest": RandomForestClassifier(
        n_estimators=500,
        random_state=42,
        max_features="sqrt",
        min_samples_leaf=20,
    ),
}

kf = KFold(n_splits=5, shuffle=True, random_state=42)

rows = []

for name, model in models.items():
    fold_losses = []
    fold_thresholds = []

    for tr, te in kf.split(X):
        model.fit(X.iloc[tr], y.iloc[tr])
        p = model.predict_proba(X.iloc[te])[:, 1]

        thr, loss = best_threshold(y.iloc[te].values, p, FP_COST, FN_COST)
        fold_losses.append(loss)
        fold_thresholds.append(thr)

    rows.append({
        "model": name,
        "avg_expected_loss": np.mean(fold_losses),
        "avg_optimal_threshold": np.mean(fold_thresholds),
    })

pd.DataFrame(rows).sort_values("avg_expected_loss")


,model,avg_expected_loss,avg_optimal_threshold
1,Random Forest,0.545326,0.102
0,Logit,0.624770,0.106


In [16]:
# CELL 14 — PART III (CORRECTED): Holdout confusion matrix using the WINNER + its CV threshold

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, roc_auc_score, brier_score_loss
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Load data
df = pd.read_csv("bisnode_fastgrowth_2012.csv")
y = df["fast_growth"]

X = df.drop(
    columns=["fast_growth", "comp_id", "year", "g1_ln_sales", "g2_ln_sales"],
    errors="ignore"
)
X = X.select_dtypes(include=[np.number])
X = X.fillna(X.mean(numeric_only=True))

# Holdout split
X_tr, X_ho, y_tr, y_ho = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ---- SET THESE TWO MANUALLY from Cell 13 ----
WINNER = "Random Forest"   # e.g. "Logit" or "Random Forest"
OPT_THR = 0.30             # paste avg_optimal_threshold for the winner from Cell 13
# --------------------------------------------

models = {
    "Logit": LogisticRegression(max_iter=2000),
    "Random Forest": RandomForestClassifier(
        n_estimators=500, random_state=42, max_features="sqrt", min_samples_leaf=20
    ),
}

best_model = models[WINNER]
best_model.fit(X_tr, y_tr)

p_ho = best_model.predict_proba(X_ho)[:, 1]
y_hat = (p_ho >= OPT_THR).astype(int)

cm = confusion_matrix(y_ho, y_hat, labels=[0,1])
cm_df = pd.DataFrame(
    cm,
    index=["Actual: not fast-growth", "Actual: fast-growth"],
    columns=["Pred: not fast-growth", "Pred: fast-growth"],
)

print(cm_df)
print("\nHoldout AUC:", round(roc_auc_score(y_ho, p_ho), 3))
print("Holdout Brier:", round(brier_score_loss(y_ho, p_ho), 3))
print("Threshold used:", OPT_THR)


                         Pred: not fast-growth  Pred: fast-growth
Actual: not fast-growth                   3238                168
Actual: fast-growth                        280                 98

Holdout AUC: 0.755
Holdout Brier: 0.081
Threshold used: 0.3


In [17]:
# CELL 15 — TASK 2: Manufacturing vs Services (same model, same loss)

from sklearn.model_selection import KFold
from sklearn.metrics import roc_auc_score, brier_score_loss, confusion_matrix
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

# Load data
df = pd.read_csv("bisnode_fastgrowth_2012.csv")

# Use SAME feature set as before (no leakage)
X_all = df.drop(
    columns=["fast_growth", "comp_id", "year", "g1_ln_sales", "g2_ln_sales"],
    errors="ignore"
)
X_all = X_all.select_dtypes(include=[np.number])
X_all = X_all.fillna(X_all.mean(numeric_only=True))

y_all = df["fast_growth"]

# SAME business loss
FP_COST = 1
FN_COST = 10

def run_group(name, mask):
    X = X_all.loc[mask]
    y = y_all.loc[mask]

    kf = KFold(n_splits=5, shuffle=True, random_state=42)

    aucs, losses = [], []

    for tr, te in kf.split(X):
        model = RandomForestClassifier(
            n_estimators=500,
            random_state=42,
            max_features="sqrt",
            min_samples_leaf=20,
        )
        model.fit(X.iloc[tr], y.iloc[tr])
        p = model.predict_proba(X.iloc[te])[:, 1]

        aucs.append(roc_auc_score(y.iloc[te], p))

        # expected loss at fixed threshold (use same 0.3 as before)
        y_hat = (p >= 0.30).astype(int)
        tn, fp, fn, tp = confusion_matrix(y.iloc[te], y_hat, labels=[0,1]).ravel()
        losses.append((fp * FP_COST + fn * FN_COST) / len(y.iloc[te]))

    return {
        "group": name,
        "n_obs": len(y),
        "cv_auc": np.mean(aucs),
        "cv_expected_loss": np.mean(losses),
        "fast_growth_rate": y.mean(),
    }

# ---- DEFINE GROUPS (adjust if your codes differ) ----
manufacturing_mask = df["ind"] == 1
services_mask = df["ind"] == 2
# ----------------------------------------------------

results = [
    run_group("Manufacturing", manufacturing_mask),
    run_group("Services", services_mask),
]

pd.DataFrame(results)


,group,n_obs,cv_auc,cv_expected_loss,fast_growth_rate
0,Manufacturing,419,0.791669,1.019449,0.140811
1,Services,5270,0.720694,0.870588,0.089943


Applying the same classification model and loss function separately to manufacturing and service firms reveals substantial heterogeneity in predictive performance. The model performs notably better for manufacturing firms (AUC = 0.79) than for service firms (AUC = 0.72), suggesting that fast growth in manufacturing is more strongly linked to observable balance-sheet and firm-structure variables. However, expected loss is higher in manufacturing, driven by a higher prevalence of fast-growing firms and the asymmetric cost assigned to false negatives. In contrast, growth in services appears harder to predict using financial data alone, consistent with the greater role of intangible assets and demand-side factors. These results indicate that while a unified modeling framework is feasible, sector-specific considerations remain important when evaluating model usefulness.